# 随机森林如何降低方差？

**面试回答：**随机森林对样本 bootstrap、对特征随机子采样，训练多棵高方差树后投票；降低的是树之间相关性和集成方差。

## 真实案例

商家拒单预测使用距离、备餐时间和雨天标记，单棵阈值树易被少量天气样本左右。

In [1]:
import numpy as np  # 导入 NumPy 手写随机森林树桩。
rng=np.random.default_rng(3)  # 固定随机数生成器。
shop=np.array(['M01','M02','M03','M04','M05','M06','M07','M08','V01','V02'])  # 构造商家订单编号。
x=np.array([[1,5,0],[2,7,0],[3,10,1],[4,9,0],[5,16,1],[6,18,1],[7,20,0],[8,22,1],[3,14,1],[7,12,0]],dtype=float)  # 记录距离、备餐和雨天。
y=np.array([0,0,0,0,1,1,1,1,1,0])  # 标记拒单。
print('订单 | 距离 | 备餐 | 雨天 | 拒单')  # 输出表头。
for n,row,c in zip(shop,x,y):  # 展示订单。
    print(n,row.tolist(),c)  # 输出样本。

订单 | 距离 | 备餐 | 雨天 | 拒单
M01 [1.0, 5.0, 0.0] 0
M02 [2.0, 7.0, 0.0] 0
M03 [3.0, 10.0, 1.0] 0
M04 [4.0, 9.0, 0.0] 0
M05 [5.0, 16.0, 1.0] 1
M06 [6.0, 18.0, 1.0] 1
M07 [7.0, 20.0, 0.0] 1
M08 [8.0, 22.0, 1.0] 1
V01 [3.0, 14.0, 1.0] 1
V02 [7.0, 12.0, 0.0] 0


## Baseline / 基线

单树桩只按备餐时间切分。

In [2]:
train=np.arange(8)  # 选择训练订单。
valid=np.arange(8,10)  # 选择验证订单。
baseline=(x[valid,1]>13).astype(int)  # 构造单特征树桩。
baseline_acc=float(np.mean(baseline==y[valid]))  # 计算基线准确率。
print('单树桩预测:',baseline.tolist(),baseline_acc)  # 输出基线。

单树桩预测: [1, 0] 1.0


In [3]:
def make_stump(data,target,feature):  # 定义单特征最优阈值树桩。
    candidates=np.unique(data[:,feature])[:-1]  # 生成候选阈值。
    scores=[np.mean(np.where(data[:,feature]<=t,int(target[data[:,feature]<=t].mean()>=.5),int(target[data[:,feature]>t].mean()>=.5))==target) for t in candidates]  # 计算每个阈值训练正确率。
    threshold=candidates[int(np.argmax(scores))]  # 选择训练正确率最高阈值。
    left=int(target[data[:,feature]<=threshold].mean()>=.5)  # 生成左叶投票类别。
    right=int(target[data[:,feature]>threshold].mean()>=.5)  # 生成右叶投票类别。
    return feature,threshold,left,right  # 返回树桩规则。
def stump_predict(data,tree):  # 定义树桩预测函数。
    feature,threshold,left,right=tree  # 解包树桩参数。
    return np.where(data[:,feature]<=threshold,left,right)  # 返回树桩分类。
trees=[]  # 创建森林树列表。
for seed in range(15):  # 训练十五棵随机树桩。
    sample=rng.integers(0,len(train),len(train))  # bootstrap 重采样训练订单位置。
    feature=int(rng.integers(0,x.shape[1]))  # 随机选择一列特征。
    trees.append(make_stump(x[train][sample],y[train][sample],feature))  # 训练当前随机树桩。
votes=np.column_stack([stump_predict(x[valid],tree) for tree in trees])  # 收集每棵树的验证投票。
pred=(votes.mean(axis=1)>=.5).astype(int)  # 多数投票形成森林结果。
acc=float(np.mean(pred==y[valid]))  # 计算森林准确率。
print('前五棵树:',trees[:5])  # 输出随机规则中间量。
print('验证投票矩阵:',votes.tolist())  # 输出树间分歧。

前五棵树: [(0, 2.0, 0, 1), (2, 0.0, 0, 1), (1, 9.0, 0, 1), (1, 10.0, 0, 1), (2, 0.0, 0, 1)]
验证投票矩阵: [[1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1], [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1]]


## 结果解读

单树会随 bootstrap 样本改变；森林平均这些不稳定规则，投票比例也能作为模型分歧信号，但不是校准概率。

In [4]:
print('订单 | 真实 | 单树 | 森林 | 赞成树数')  # 输出结果表头。
for i,index in enumerate(valid):  # 逐条展示集成结果。
    print(shop[index],y[index],baseline[i],pred[i],int(votes[i].sum()))  # 输出投票证据。
print('生产差距：完整森林需多层树、OOB、特征重要性偏差审计和延迟/体积预算。')  # 说明简化边界。

订单 | 真实 | 单树 | 森林 | 赞成树数
V01 1 1 1 12
V02 0 0 1 12
生产差距：完整森林需多层树、OOB、特征重要性偏差审计和延迟/体积预算。


## 失败案例与修复

若所有树使用同一全量数据和同一特征，投票完全相关，几乎没有降方差；修复是 bootstrap 与特征子采样。

In [5]:
same_tree=make_stump(x[train],y[train],1)  # 构造不带随机性的同一树桩。
same_votes=np.column_stack([stump_predict(x[valid],same_tree) for _ in range(15)])  # 复制相同树的投票。
print('失败：相同树投票方差=',float(same_votes.var()))  # 输出完全相关的投票方差。
print('修复：随机森林投票方差=',float(votes.var()))  # 输出随机树间差异。
print('注意：随机性降低方差但也可能降低单树强度。')  # 解释取舍。

失败：相同树投票方差= 0.0
修复：随机森林投票方差= 0.16
注意：随机性降低方差但也可能降低单树强度。


In [6]:
assert len(shop)>=5  # 保护样本数。
assert len(trees)==15  # 保护森林树数。
assert 0.0<=acc<=1.0  # 保护森林指标处于合法范围而不虚假宣称小样本必然更强。
assert same_votes.var()==0  # 保护失败案例确实完全相关。